# Notebook 05 — Process-Aware Rescue Simulation

## Goal

Extend the decision engine from Notebook04 by introducing a **virtual process optimization layer**.

In real CLD workflows, clone performance is not fixed.  
It depends strongly on:

- media composition
- feeding strategy
- metabolic control
- process conditions

Some clones—especially **rescue candidates**—may improve significantly under optimized conditions.

---

## What this notebook does

This notebook introduces a simplified simulation of:

> "What if rescue clones benefit from process optimization?"

We:

1. Apply a **virtual optimization effect** to rescue clones
2. Recompute predicted performance under optimized conditions
3. Re-run ranking and selection
4. Compare results before vs after optimization

---

## Key idea

- Pass clones = already strong → minimal improvement
- Rescue clones = process-sensitive → larger potential gain

---

## Pipeline

Notebook03 → prediction  
Notebook04 → decision (baseline)  
Notebook05 → **process-aware re-ranking**

---

## Important note

This is a **simulation layer**, not a real bioprocess model.

It approximates:
- productivity gains
- aggregation reduction
- stability improvements

based on rescue potential.

## Step 1 — Load prediction table

We load the same prediction table used in Notebook04.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

scenario = "legacy"
n_clones = 5000

PRED_PATH = Path("../data/synthetic/processed") / f"predictions_03b_qp_{n_clones}_{scenario}.csv"

df = pd.read_csv(PRED_PATH)
print("Shape:", df.shape)
df.head()

Shape: (1000, 19)


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob,rescue_upside_qp,rescue_stability_band,rescue_quality_band,rescue_aggressive_penalty,rescue_not_already_pass,pred_rescue_score,pred_rescue_label
0,CLONE_1502,0.498165,2.673129e-08,8.435996,0.238692,0,0.206868,3.215605e-08,4.404966,1,0.001494,0.003219,0.000312,0.506117,0.981713,0.996769,0.710487,0.568590,0
1,CLONE_2587,0.373417,8.407121e-08,3.130765,0.464397,0,0.600740,5.393417e-08,3.662670,0,0.007656,0.000000,0.013124,0.921942,0.000000,1.000000,0.435364,0.416664,0
2,CLONE_2654,0.461606,2.770586e-08,7.582349,0.256485,0,0.156409,3.459109e-08,10.144181,1,0.001523,0.004162,0.000529,0.627981,0.737814,0.995823,0.688798,0.538817,0
3,CLONE_1056,0.348761,7.746555e-08,6.885601,0.507813,1,0.465880,6.392818e-08,2.383546,0,0.012355,0.000000,0.011648,0.991742,0.538743,1.000000,0.382442,0.594558,0
4,CLONE_0706,0.644735,1.744327e-07,8.275030,0.018172,0,0.678633,1.526521e-07,5.032915,0,0.004539,0.039329,0.033316,0.017551,0.935723,0.960529,0.979289,0.407241,0


## Step 2 — Ensure rescue columns exist

In [2]:
if "pred_rescue_score" not in df.columns:
    df["pred_rescue_score"] = 0.0

if "pred_rescue_label" not in df.columns:
    df["pred_rescue_label"] = 0

## Step 3 — Helper functions

In [3]:
def z(s):
    s = pd.Series(s).astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

## Step 4 — Recreate baseline bucket assignment

We reuse the same biosimilar decision thresholds.

In [4]:
# Biosimilar thresholds (same as Notebook04 baseline)
BIO_AGG_PASS_MAX = 15.0
BIO_STABLE_PASS_MIN = 0.50

BIO_AGG_RESCUE_MAX = 18.0
BIO_STABLE_RESCUE_MIN = 0.25
BIO_RESCUE_SCORE_MIN = df["pred_rescue_score"].quantile(0.70)

bio = df.copy()

bio["bio_pass"] = (
    (bio["pred_late_agg"] <= BIO_AGG_PASS_MAX) &
    (bio["pred_stable_prob"] >= BIO_STABLE_PASS_MIN)
)

bio["bio_rescue"] = (
    (~bio["bio_pass"]) &
    (bio["pred_rescue_score"] >= BIO_RESCUE_SCORE_MIN) &
    (bio["pred_late_agg"] <= BIO_AGG_RESCUE_MAX) &
    (bio["pred_stable_prob"] >= BIO_STABLE_RESCUE_MIN)
)

bio["bio_bucket"] = "fail"
bio.loc[bio["bio_rescue"], "bio_bucket"] = "rescue"
bio.loc[bio["bio_pass"], "bio_bucket"] = "pass"

bio["bio_bucket"].value_counts()

bio_bucket
fail      758
rescue    160
pass       82
Name: count, dtype: int64

## Step 5 — Apply virtual process optimization

We simulate that **rescue clones benefit from process optimization**.

Assumptions:
- productivity increases
- aggregation decreases
- stability improves

Effect strength is proportional to `pred_rescue_score`.

In [5]:
alpha_qp = 0.8   # productivity gain
beta_agg = 0.4   # aggregation reduction
gamma_stab = 0.3 # stability improvement

opt = bio.copy()

mask = opt["bio_bucket"] == "rescue"

# Productivity boost
opt["opt_pred_late_qp"] = opt["pred_late_qp"]
opt.loc[mask, "opt_pred_late_qp"] = (
    opt.loc[mask, "pred_late_qp"] *
    (1 + alpha_qp * opt.loc[mask, "pred_rescue_score"])
)

# Aggregation reduction
opt["opt_pred_late_agg"] = opt["pred_late_agg"]
opt.loc[mask, "opt_pred_late_agg"] = (
    opt.loc[mask, "pred_late_agg"] *
    (1 - beta_agg * opt.loc[mask, "pred_rescue_score"])
)

# Stability improvement
opt["opt_pred_stable_prob"] = opt["pred_stable_prob"]
opt.loc[mask, "opt_pred_stable_prob"] = (
    opt.loc[mask, "pred_stable_prob"] +
    gamma_stab * opt.loc[mask, "pred_rescue_score"]
)

## Step 6 — Re-ranking after optimization

In [6]:
# --------------------------------------------------
# Step 6 — Process-aware scoring design
# --------------------------------------------------
# Goal:
# Make optimization effect influence ranking more directly.
#
# Instead of only adding adjusted qP / aggregation into a normal score,
# we compute:
# 1) baseline score
# 2) process gain score
# 3) multiplicative process-aware score

# Baseline score using original predictions
opt["base_score"] = (
    z(opt["pred_late_qp"])
    - 0.5 * z(opt["pred_qp_drop"])
    - 0.2 * z(opt["pred_late_agg"])
    + 0.3 * z(opt["pred_rescue_score"])
)

# Optimized score using virtual process-adjusted predictions
opt["opt_raw_score"] = (
    z(opt["opt_pred_late_qp"])
    - 0.5 * z(opt["pred_qp_drop"])
    - 0.2 * z(opt["opt_pred_late_agg"])
    + 0.3 * z(opt["pred_rescue_score"])
)

# Process gain components
opt["process_gain_qp"] = (
    opt["opt_pred_late_qp"] - opt["pred_late_qp"]
) / (opt["pred_late_qp"].abs() + 1e-12)

opt["process_gain_agg"] = (
    opt["pred_late_agg"] - opt["opt_pred_late_agg"]
) / (opt["pred_late_agg"].abs() + 1e-12)

opt["process_gain_stability"] = (
    opt["opt_pred_stable_prob"] - opt["pred_stable_prob"]
)

# Combined process responsiveness score
opt["process_gain_score"] = (
    0.5 * z(opt["process_gain_qp"])
    + 0.3 * z(opt["process_gain_agg"])
    + 0.2 * z(opt["process_gain_stability"])
)

# Only rescue clones should receive process-aware multiplier
opt["process_multiplier"] = 1.0
rescue_mask = opt["bio_bucket"] == "rescue"

opt.loc[rescue_mask, "process_multiplier"] = (
    1.0 + 0.35 * opt.loc[rescue_mask, "pred_rescue_score"]
)

# Final process-aware score
# This preserves baseline ranking but allows high-rescue-potential clones to move upward.
opt["opt_final_score"] = (
    opt["opt_raw_score"] * opt["process_multiplier"]
    + 0.5 * opt["process_gain_score"]
)

display(opt[[
    "clone_id",
    "bio_bucket",
    "base_score",
    "opt_raw_score",
    "process_gain_score",
    "process_multiplier",
    "opt_final_score",
    "pred_rescue_score",
    "pred_late_qp",
    "opt_pred_late_qp",
    "pred_late_agg",
    "opt_pred_late_agg",
    "pred_stable_prob",
    "opt_pred_stable_prob",
]].sort_values("opt_final_score", ascending=False).head(15))

,clone_id,bio_bucket,base_score,opt_raw_score,process_gain_score,process_multiplier,opt_final_score,pred_rescue_score,pred_late_qp,opt_pred_late_qp,pred_late_agg,opt_pred_late_agg,pred_stable_prob,opt_pred_stable_prob
180,CLONE_4625,pass,13.368297,13.238889,-0.434375,1.000000,13.021702,1.000000,4.500575e-06,4.500575e-06,8.670837,8.670837,0.560659,0.560659
621,CLONE_3254,pass,12.803429,12.679009,-0.434375,1.000000,12.461821,0.787107,4.328425e-06,4.328425e-06,8.356089,8.356089,0.821561,0.821561
505,CLONE_0048,fail,10.373058,10.230845,-0.434375,1.000000,10.013657,0.701157,4.109802e-06,4.109802e-06,10.686979,10.686979,0.199181,0.199181
524,CLONE_2171,fail,9.742342,9.620614,-0.434375,1.000000,9.403426,0.711942,4.044897e-06,4.044897e-06,8.462717,8.462717,0.114915,0.114915
986,CLONE_2776,fail,9.123223,9.018506,-0.434375,1.000000,8.801319,0.577185,3.750558e-06,3.750558e-06,6.964564,6.964564,0.168391,0.168391
730,CLONE_4878,fail,8.995326,8.869838,-0.434375,1.000000,8.652650,0.731989,3.511339e-06,3.511339e-06,9.662363,9.662363,0.172336,0.172336
314,CLONE_1619,pass,7.731381,7.672899,-0.434375,1.000000,7.455711,0.358295,2.630590e-06,2.630590e-06,3.356785,3.356785,0.731156,0.731156
63,CLONE_3863,fail,6.706221,6.602926,-0.434375,1.000000,6.385739,0.581613,3.052754e-06,3.052754e-06,7.816017,7.816017,0.057090,0.057090
898,CLONE_3393,rescue,1.965760,3.134177,2.739060,1.252957,5.296519,0.722734,6.424109e-07,1.013845e-06,8.864235,6.301641,0.295613,0.512434
188,CLONE_0531,rescue,2.072074,3.301999,2.366752,1.223280,5.222645,0.637943,7.667192e-07,1.158017e-06,11.299661,8.416247,0.449029,0.640412


## Step 7 — Compare Top10 (baseline vs optimized)

In [7]:
bio["base_score"] = opt["base_score"].copy()

# Baseline
bio["base_score"] = (
    z(bio["pred_late_qp"])
    - 0.5 * z(bio["pred_qp_drop"])
    - 0.2 * z(bio["pred_late_agg"])
    + 0.3 * z(bio["pred_rescue_score"])
)

base_top10 = bio.sort_values("base_score", ascending=False).head(10)
opt_top10 = opt.sort_values("opt_final_score", ascending=False).head(10)

print("Baseline Top10 rescue count:", (base_top10["bio_bucket"] == "rescue").sum())
print("Optimized Top10 rescue count:", (opt_top10["bio_bucket"] == "rescue").sum())

base_ids = set(base_top10["clone_id"])
opt_ids = set(opt_top10["clone_id"])

print("Overlap:", len(base_ids & opt_ids))

Baseline Top10 rescue count: 0
Optimized Top10 rescue count: 2
Overlap: 8


## Step 8 — Rescue survival analysis

We evaluate whether rescue clones improve their ranking under process optimization.

In [8]:
rescue_before = base_top10[base_top10["bio_bucket"] == "rescue"]
rescue_after = opt_top10[opt_top10["bio_bucket"] == "rescue"]

print("Rescue in baseline:", len(rescue_before))
print("Rescue after optimization:", len(rescue_after))

Rescue in baseline: 0
Rescue after optimization: 2


## Step 9 — Utility comparison (optional)

In [9]:
def topk_overlap(true_scores, pred_scores, k):
    true_top = set(pd.Series(true_scores).nlargest(k).index)
    pred_top = set(pd.Series(pred_scores).nlargest(k).index)
    return len(true_top & pred_top) / k

# Compute true utility for both baseline and optimized tables
bio["true_util"] = (
    z(bio["true_late_qp"])
    - z(bio["true_qp_drop"])
    - 0.2 * z(bio["true_late_agg"])
)

opt["true_util"] = (
    z(opt["true_late_qp"])
    - z(opt["true_qp_drop"])
    - 0.2 * z(opt["true_late_agg"])
)

base_overlap = topk_overlap(bio["true_util"], bio["base_score"], 10)
opt_overlap = topk_overlap(opt["true_util"], opt["opt_final_score"], 10)

print("Utility overlap (baseline):", base_overlap)
print("Utility overlap (optimized):", opt_overlap)

Utility overlap (baseline): 0.3
Utility overlap (optimized): 0.3


## Interpretation of Notebook 05

This notebook introduces a **process-aware simulation layer** on top of the decision engine.

### Current baseline finding

Under the current optimization-effect assumptions, rescue clones improve virtually but do not yet enter the optimized Top-10 set.

This suggests that either:

- the assumed process optimization effect is too weak, or
- rescue clones start too far below pass clones in predicted performance, or
- additional process-specific features are needed to model true rescue potential.

Therefore, Notebook05 should be interpreted as a baseline sensitivity test rather than a completed process-optimization model.

---

### Conceptual meaning

This demonstrates that:

> clone selection should not be based solely on static predictions.

Instead, selection should consider:

- intrinsic clone performance
- process sensitivity
- optimization potential

---

### Practical implication

- Top-3 production clones remain stable (safe selection)
- Top-10 pool becomes more diverse
- Rescue clones represent **process-optimization opportunities**

---

### Biological rationale

In industrial CHO systems:

- productivity depends strongly on feeding strategy
- media composition affects metabolism and protein quality
- process optimization can significantly increase yield

Therefore, rescue clones may not be inferior—  
they may simply require better process conditions.

---

### Conclusion

Notebook05 bridges:

Prediction → Decision → Process

This is a critical step toward:

> **process-aware clone selection systems**